CODE DUMMY DATA GENERATOR INI HANYA UNTUK SEKALI RUN AJA, KARENA MENYESUAIKAN STUDY CASE DIMANA KE 3 TABLE (CUSTOMERS_RAW, SALES, AFTER_SALES) SUDAH EXSITING DI DATABASE MYSQL

Generate Data Tabel customers_raw

In [ ]:
import random
from faker import Faker
from datetime import datetime

fake = Faker()

def random_dob():
    formats = [
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%d/%m/%Y",
        None
    ]

    chosen_format = random.choice(formats)

    if chosen_format is None:
        return None

    date = fake.date_of_birth(minimum_age=15, maximum_age=100)
    return date.strftime(chosen_format)

def random_name():
    if random.random() < 0.2:
        return fake.company()
    else:
        return fake.first_name()

def random_created_at():
    dt = fake.date_time_this_year()

    # generate millisecond (0–999)
    ms = random.randint(0, 999)

    # format ke string + millisecond
    return dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{ms:03d}'

def generate_data(n=1000):
    data = []

    for i in range(7, n+1):
        name = random_name()
        dob = random_dob()
        created_at = random_created_at()

        data.append((i, name, dob, created_at))

    return data


# Example output
data = generate_data(1000)

for row in data:
    print(row)

INJECT TO TABLE STG_CUSTOMER_RAW

In [ ]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_CUSTOMERS_RAW (id, name, dob, created_at)
    VALUES (%s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DATA TABLE SALES_RAW

In [ ]:
import random
from datetime import datetime, timedelta

# SAMPLE MASTER DATA
model_price_map = {
    'RAIZA': '350.000.000',
    'RANGGO': '430.000.000',
    'INNAVO': '600.000.000',
    'VELOS': '390.000.000'
}

models = list(model_price_map.keys())

# SIMULASI customer created_at
def generate_customer_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)
    for i in range(1, n+1):
        random_days = random.randint(0, 60)
        random_seconds = random.randint(0, 86400)
        dt = base_date + timedelta(days=random_days, seconds=random_seconds)
        data[i] = dt
    return data

customer_created_at = generate_customer_master(1000)

# GENERATE VIN UNIQUE
used_vins = set()
def generate_unique_vin():
    while True:
        vin = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10))
        if vin not in used_vins:
            used_vins.add(vin)
            return vin

# Generate data
def generate_data(n=2000):
    data = []
    for _ in range(n):
        customer_id = random.randint(1, 1000)
        cust_dt = customer_created_at[customer_id]

        # Gunakan satu datetime untuk invoice_date & created_at
        invoice_created_dt = cust_dt + timedelta(days=random.randint(0, 30), seconds=random.randint(0, 86400))

        vin = generate_unique_vin()
        model = random.choice(models)
        price = model_price_map[model]

        created_at = invoice_created_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(invoice_created_dt.microsecond/1000):03d}'
        invoice_date = invoice_created_dt.strftime('%Y-%m-%d')

        data.append((
            vin,
            customer_id,
            model,
            invoice_date,
            price,
            created_at
        ))

    return data

# GENERATE DATA
data = generate_data()

# PRINT SAMPLE
for row in data[:10]:
    print(row)

INJECT TO TABLE SALES_RAW

In [ ]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_SALES_RAW (
    vin,
    customer_id,
    model,
    invoice_date,
    price,
    created_at)
    VALUES (%s, %s, %s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DUMMY DATA AFTER SALES RAW

In [ ]:
import random
import string
from datetime import datetime, timedelta

# GENERATOR VIN UNIK
used_vins = set()
def generate_unique_vin():
    while True:
        vin = ''.join(random.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789', k=10))
        if vin not in used_vins:
            used_vins.add(vin)
            return vin

# MASTER DATA
model_map = {
    'JIS8135SAD': 'RAIZA',
    'MAS8160POE': 'RANGGO',
    'JLK1368KDE': 'INNAVO',
    'JLK1869KDF': 'VELOS',
    'JLK1962KOP': 'VELOS'
}
service_types = ['BP', 'PM', 'GR']

# SIMULASI SALES_RAW created_at
def generate_sales_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)
    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 180),
            seconds=random.randint(0, 86400)
        )
        data[i] = dt
    return data

sales_created_at = generate_sales_master(1000)

# GENERATE SERVICE TICKET
def generate_ticket():
    number = random.randint(100, 999)
    letters = ''.join(random.choices(string.ascii_lowercase, k=3))
    suffix = random.randint(1, 9)
    return f"T{number}-{letters}{suffix}"

# GENERATE DATETIME AFTER SALES CREATED_AT
def random_datetime_after(base_dt):
    max_date = datetime(2026, 12, 31)
    delta = max_date - base_dt
    random_seconds = random.randint(1, int(delta.total_seconds()))
    new_dt = base_dt + timedelta(seconds=random_seconds)
    ms = random.randint(0, 999)
    new_dt = new_dt.replace(microsecond=ms * 1000)
    return new_dt

# GENERATE DATA
def generate_data(n=500):
    data = []

    for _ in range(n):
        customer_id = random.randint(1, 1000)
        sales_dt = sales_created_at[customer_id]

        created_at_dt = random_datetime_after(sales_dt)
        service_date = created_at_dt.strftime('%Y-%m-%d')
        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(created_at_dt.microsecond/1000):03d}'

        vin = generate_unique_vin()
        # assign random model untuk VIN baru
        model = random.choice(['RAIZA', 'RANGGO', 'INNAVO', 'VELOS'])
        model_map[vin] = model  # simpan mapping VIN -> model
        service_type = random.choice(service_types)
        ticket = generate_ticket()

        data.append((
            ticket,
            vin,
            customer_id,
            model,
            service_date,
            service_type,
            created_at
        ))

    return data

data = generate_data(1000)

for row in data:
    print(row)

In [ ]:
import random
import string
from datetime import datetime, timedelta

# MASTER DATA
vin_list = ['JIS8135SAD', 'MAS8160POE', 'JLK1368KDE', 'JLK1869KDF', 'JLK1962KOP']

model_map = {
    'JIS8135SAD': 'RAIZA',
    'MAS8160POE': 'RANGGO',
    'JLK1368KDE': 'INNAVO',
    'JLK1869KDF': 'VELOS',
    'JLK1962KOP': 'VELOS'
}

service_types = ['BP', 'PM', 'GR']


# SIMULASI SALES_RAW created_at
def generate_sales_master(n=1000):
    data = {}
    base_date = datetime(2025, 1, 1)

    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 180),
            seconds=random.randint(0, 86400)
        )
        data[i] = dt

    return data

sales_created_at = generate_sales_master(1000)


# GENERATE SERVICE TICKET
def generate_ticket():
    number = random.randint(100, 999)
    letters = ''.join(random.choices(string.ascii_lowercase, k=3))
    suffix = random.randint(1, 9)
    return f"T{number}-{letters}{suffix}"


# GENERATE DATETIME AFTER SALES CREATED_AT
def random_datetime_after(base_dt):
    max_date = datetime(2026, 12, 31)

    delta = max_date - base_dt
    random_seconds = random.randint(1, int(delta.total_seconds()))

    new_dt = base_dt + timedelta(seconds=random_seconds)

    # millisecond
    ms = random.randint(0, 999)
    new_dt = new_dt.replace(microsecond=ms * 1000)

    return new_dt


def generate_data(n=1000):
    data = []

    for _ in range(n):
        customer_id = random.randint(1, 1000)

        # ambil created_at dari sales_raw
        sales_dt = sales_created_at[customer_id]

        # generate service datetime (HARUS setelah sales)
        created_at_dt = random_datetime_after(sales_dt)

        service_date = created_at_dt.strftime('%Y-%m-%d')

        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + f'.{int(created_at_dt.microsecond/1000):03d}'

        vin = random.choice(vin_list)
        model = model_map[vin]
        service_type = random.choice(service_types)
        ticket = generate_ticket()

        data.append((
            ticket,
            vin,
            customer_id,
            model,
            service_date,
            service_type,
            created_at
        ))

    return data


# GENERATE
data = generate_data(500)

# PRINT
for row in data:
    print(row)

INJECT TO TABLE STG_AFTER_SALES_RAW

In [ ]:
import mysql.connector
from getpass import getpass

password = getpass("Masukkan password MySQL: ")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password=password,
    database="mysql-dwh-dev"
)

cursor = conn.cursor()

# data = generate_data(1000)

cursor.executemany("""
    INSERT INTO STG_AFTER_SALES_RAW (
    service_ticket,
    vin,
    customer_id,
    model,
    service_date,
    service_type,
    created_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
""", data)

conn.commit()
cursor.close()
conn.close()

DATA TABLE CUSTOMER_ADDRESS

In [ ]:
import random
from faker import Faker
from datetime import datetime, timedelta
import os

fake = Faker('id_ID')

DELIMITER = "|@~"

# ambil tanggal hari ini
today_str = datetime.now().strftime('%Y%m%d')

# path dengan nama file harian
file_path = fr"D:\Git\Repository\End-To-End-Data-Pipeline-Using-Airflow-Docker-MySQL-CSV\data\customer_address_{today_str}.csv"


# SIMULASI customers_raw
def generate_customer_master(n=1000):
    data = {}
    base_date = datetime(2026, 1, 1)

    for i in range(1, n+1):
        dt = base_date + timedelta(
            days=random.randint(0, 90),
            seconds=random.randint(0, 86400)
        )
        ms = random.randint(0, 999)
        dt = dt.replace(microsecond=ms * 1000)
        data[i] = dt

    return data


customer_created_at = generate_customer_master(1000)


def generate_data(n=1000):
    if n > 1000:
        raise ValueError("Max 1000 rows")

    data = []
    customer_ids = random.sample(range(1, 1001), n)

    for i, cust_id in enumerate(customer_ids, start=1):
        created_at_dt = customer_created_at[cust_id]

        created_at = created_at_dt.strftime('%Y-%m-%d %H:%M:%S') + \
                     f'.{int(created_at_dt.microsecond/1000):03d}'

        address = fake.street_address().replace('\n', ' ')
        city = fake.city()
        province = fake.state()

        data.append([
            str(i),
            str(cust_id),
            address,
            city,
            province,
            created_at
        ])

    return data


# GENERATE
data = generate_data(1000)

# SAVE FILE
os.makedirs(os.path.dirname(file_path), exist_ok=True)

with open(file_path, 'w', encoding='utf-8') as f:
    # header
    f.write(DELIMITER.join(["id","customer_id","address","city","province","created_at"]) + "\n")

    # rows
    for row in data:
        f.write(DELIMITER.join(row) + "\n")

print(f"File berhasil dibuat: {file_path}")